# **Healthcare Readmission KPI Analysis**
##[](url) **CMS Hospital Readmissions Reduction Program (HRRP)**

This notebook contains initial exploration and preparation of CMS hospital readmission data for healthcare KPI reporting, dashboard visualization, and benchmarking analysis using Databricks SQL.

## **Dataset Overview**



### Dataset
This project uses the CMS Hospital Readmissions Reduction Program (HRRP) dataset, which contains publicly reported hospital quality measures related to excess hospital readmissions across multiple medical conditions.

### Source
Source: Centers for Medicare & Medicaid Services (CMS) public reporting data from data.cms.gov.

### Reporting Period
The dataset reflects CMS rolling reporting periods spanning July 2021 through June 2024 rather than traditional calendar years. 

### Measure Definitions
The measures represent hospital-level excess readmission performance for selected conditions and procedures, including:
- Heart Failure
- COPD
- Pneumonia
- Heart Attack
- Hip/Knee Replacement

The dataset also includes benchmarking metrics, payment reduction information, hospital identifiers, geographic information, and CMS reporting footnotes related to data quality or reporting limitations.

In [0]:
%sql
Select *
From cms_readmissions
Limit 20

Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137,null,0.9531,10.3960,10.9078,13,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273,null,0.9370,13.2998,14.1948,33,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122,null,0.9823,16.6384,16.9389,19,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507,null,0.9871,15.7529,15.9591,79,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HF-HRRP,653,null,1.0233,20.5695,20.1010,136,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-CABG-HRRP,N/A,5,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-AMI-HRRP,N/A,1,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-COPD-HRRP,132,null,0.9308,16.8541,18.1080,17,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-HF-HRRP,164,null,1.0087,20.9512,20.7700,35,07/01/2021,2024-06-30


In [0]:
%sql

SELECT COUNT(*) AS row_count
FROM cms_readmissions

row_count
18330


In [0]:
%sql

SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


## Rename Fields

In [0]:


%sql
CREATE OR REPLACE TABLE workspace.default.cms_readmissions_clean AS

-- Rename columns for consistency

SELECT `Facility Name` as facility_name, `Facility ID` as facility_id, state, `Measure Name` as measure_name, `Number of Discharges` as Number_of_Discharges, Footnote, `Excess Readmission Ratio` as Excess_Readmission_Ratio, `Predicted Readmission Rate`as Predicted_Readmission_Rate, `Expected Readmission Rate` as Expected_Readmission_Rate, `Number of Readmissions` as Number_of_Readmissions, `Start Date` as Start_Date, `End Date` as End_date
from workspace.default.cms_readmissions;


num_affected_rows,num_inserted_rows


In [0]:

%sql

SELECT *
FROM workspace.default.cms_readmissions_clean
LIMIT 20;

facility_name,facility_id,state,measure_name,Number_of_Discharges,Footnote,Excess_Readmission_Ratio,Predicted_Readmission_Rate,Expected_Readmission_Rate,Number_of_Readmissions,Start_Date,End_date
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137,null,0.9531,10.3960,10.9078,13,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273,null,0.9370,13.2998,14.1948,33,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122,null,0.9823,16.6384,16.9389,19,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507,null,0.9871,15.7529,15.9591,79,07/01/2021,2024-06-30
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HF-HRRP,653,null,1.0233,20.5695,20.1010,136,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-CABG-HRRP,N/A,5,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-AMI-HRRP,N/A,1,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-COPD-HRRP,132,null,0.9308,16.8541,18.1080,17,07/01/2021,2024-06-30
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-HF-HRRP,164,null,1.0087,20.9512,20.7700,35,07/01/2021,2024-06-30


Review distinct CMS readmission measures included in the dataset.

In [0]:
%sql
SELECT DISTINCT measure_name
FROM workspace.default.cms_readmissions_clean

measure_name
READM-30-HIP-KNEE-HRRP
READM-30-CABG-HRRP
READM-30-AMI-HRRP
READM-30-COPD-HRRP
READM-30-PN-HRRP
READM-30-HF-HRRP


## Measure Standardization
## 
CMS measure codes were translated into business-friendly labels for reporting and dashboard visualization.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.cms_readmissions_clean1 AS

-- Rename measures 
SELECT *,

        CASE
        WHEN measure_name = 'READM-30-HIP-KNEE-HRRP'
        THEN 'Hip/Knee Replacement'

        WHEN measure_name = 'READM-30-AMI-HRRP'
            THEN 'Heart Attack'

        WHEN measure_name = 'READM-30-CABG-HRRP'
            THEN 'Coronary Artery Bypass Graft Surgery (CABG)'

        WHEN measure_name = 'READM-30-COPD-HRRP'
            THEN 'COPD'

        WHEN measure_name = 'READM-30-PN-HRRP'
            THEN 'Pneumonia'

        WHEN measure_name = 'READM-30-HF-HRRP'
            THEN 'Heart Failure'

        ELSE measure_name
        END AS measure_display_name

FROM workspace.default.cms_readmissions_clean;




num_affected_rows,num_inserted_rows



## Data Quality Review

This section evaluates reporting footnotes, missing values, and suppressed results contained within the CMS HRRP dataset. Because CMS hospital quality reporting includes public reporting limitations and statistical reliability protections, understanding data quality indicators is important before performing KPI comparisons or dashboard analysis.

Key review areas include:
- CMS reporting footnotes
- missing or null KPI values
- suppressed or unavailable results
- low-volume reporting indicators
- partial reporting periods

Certain CMS footnotes indicate suppressed or unavailable reporting results due to small sample sizes, incomplete reporting periods, or data quality limitations. These indicators should be considered when interpreting hospital performance comparisons.

In [0]:
%sql
SELECT footnote,
       COUNT(*) As row_count
FROM workspace.default.cms_readmissions_clean1
GROUP BY footnote
ORDER BY row_count DESC

footnote,row_count
null,11343
5,3255
1,3150
29,377
7,205


Join the footnote explanations for clarity.

In [0]:
%sql

CREATE OR REPLACE TABLE  footnote_table as
SELECT `Footnote Number` as footnote_number, `Footnote Explanation` as footnote_Explanation
FROM workspace.default.footnote_appendix


num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE  cms_readmissions_clean2 as
SELECT *
FROM workspace.default.cms_readmissions_clean1
LEFT JOIN workspace.default.footnote_table
ON workspace.default.cms_readmissions_clean1.Footnote = workspace.default.footnote_table.footnote_number


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.default.cms_readmissions_clean2
LIMIT 20

facility_name,facility_id,state,measure_name,Number_of_Discharges,Footnote,Excess_Readmission_Ratio,Predicted_Readmission_Rate,Expected_Readmission_Rate,Number_of_Readmissions,Start_Date,End_date,measure_display_name,footnote_number,footnote_Explanation
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137,null,0.9531,10.3960,10.9078,13,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),null,null
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273,null,0.9370,13.2998,14.1948,33,07/01/2021,2024-06-30,Heart Attack,null,null
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122,null,0.9823,16.6384,16.9389,19,07/01/2021,2024-06-30,COPD,null,null
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507,null,0.9871,15.7529,15.9591,79,07/01/2021,2024-06-30,Pneumonia,null,null
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HF-HRRP,653,null,1.0233,20.5695,20.1010,136,07/01/2021,2024-06-30,Heart Failure,null,null
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-CABG-HRRP,N/A,5,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),5,Results are not available for this reporting period.
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-AMI-HRRP,N/A,1,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30,Heart Attack,1,The number of cases/patients is too few to report.
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-COPD-HRRP,132,null,0.9308,16.8541,18.1080,17,07/01/2021,2024-06-30,COPD,null,null
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-HF-HRRP,164,null,1.0087,20.9512,20.7700,35,07/01/2021,2024-06-30,Heart Failure,null,null


In [0]:
%sql
SELECT footnote_number, footnote_explanation
FROM cms_readmissions_clean2
WHERE footnote_number IS NOT NULL
LIMIT 20

footnote_number,footnote_explanation
5,Results are not available for this reporting period.
1,The number of cases/patients is too few to report.
1,The number of cases/patients is too few to report.
1,The number of cases/patients is too few to report.
5,Results are not available for this reporting period.
5,Results are not available for this reporting period.
1,The number of cases/patients is too few to report.
5,Results are not available for this reporting period.
1,The number of cases/patients is too few to report.
1,The number of cases/patients is too few to report.


## Review for Data Quality

Inspect missing values for Excess Readmission Ratio

In [0]:
%sql
SELECT COUNT (*)
FROM workspace.default.cms_readmissions_clean2
WHERE Excess_Readmission_Ratio IS NULL
    

COUNT(*)
0


In [0]:
%sql

DESCRIBE workspace.default.cms_readmissions_clean2

col_name,data_type,comment
facility_name,string,null
facility_id,bigint,null
state,string,null
measure_name,string,null
Number_of_Discharges,string,null
Footnote,bigint,null
Excess_Readmission_Ratio,string,null
Predicted_Readmission_Rate,string,null
Expected_Readmission_Rate,string,null
Number_of_Readmissions,string,null


In [0]:
%sql

SELECT *
FROM workspace.default.cms_readmissions_clean2
WHERE Number_of_Readmissions LIKE '%Too Few%'
LIMIT 20;

facility_name,facility_id,state,measure_name,Number_of_Discharges,Footnote,Excess_Readmission_Ratio,Predicted_Readmission_Rate,Expected_Readmission_Rate,Number_of_Readmissions,Start_Date,End_date,measure_display_name,footnote_number,footnote_Explanation
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.8602,4.6113,5.3609,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null
NORTH ALABAMA MEDICAL CENTER,10006,AL,READM-30-HIP-KNEE-HRRP,N/A,null,1.0527,6.0830,5.7783,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null
NORTH ALABAMA MEDICAL CENTER,10006,AL,READM-30-CABG-HRRP,N/A,null,1.0027,11.1697,11.1399,Too Few to Report,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),null,null
MIZELL MEMORIAL HOSPITAL,10007,AL,READM-30-COPD-HRRP,N/A,null,1.0232,16.5931,16.2168,Too Few to Report,07/01/2021,2024-06-30,COPD,null,null
CRENSHAW COMMUNITY HOSPITAL,10008,AL,READM-30-PN-HRRP,N/A,null,1.0017,13.4114,13.3883,Too Few to Report,07/01/2021,2024-06-30,Pneumonia,null,null
DEKALB REGIONAL MEDICAL CENTER,10012,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9322,4.0417,4.3357,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null
DEKALB REGIONAL MEDICAL CENTER,10012,AL,READM-30-AMI-HRRP,N/A,null,1.0565,11.4786,10.8650,Too Few to Report,07/01/2021,2024-06-30,Heart Attack,null,null
DEKALB REGIONAL MEDICAL CENTER,10012,AL,READM-30-COPD-HRRP,N/A,null,0.9934,13.8163,13.9081,Too Few to Report,07/01/2021,2024-06-30,COPD,null,null
SHELBY BAPTIST MEDICAL CENTER,10016,AL,READM-30-CABG-HRRP,N/A,null,1.0875,12.3307,11.3390,Too Few to Report,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),null,null


Records marked “Too Few to Report” were converted to NULL in numeric KPI fields to avoid treating suppressed values as zero. A separate flag was retained to preserve CMS reporting limitations for data quality analysis.

In [0]:
%sql

CREATE OR REPLACE TABLE cms_readmissions_clean3 AS

SELECT *,

    CASE
        WHEN Number_of_Readmissions = 'Too Few to Report' THEN NULL
        WHEN Number_of_Readmissions = 'N/A' THEN NULL
        ELSE CAST(Number_of_Readmissions AS BIGINT)
    END AS Number_of_Readmissions_numeric,

    CASE
        WHEN Number_of_Readmissions = 'Too Few to Report' THEN 1
        ELSE 0
    END AS readmissions_too_few_flag,

    CASE    
        WHEN excess_readmission_ratio = 'N/A'
            THEN NULL
        ELSE CAST(excess_readmission_ratio AS DOUBLE)
    END AS excess_readmission_ratio_numeric,

    CASE
        WHEN Number_of_Discharges = 'N/A' THEN NULL
        ELSE CAST(Number_of_Discharges AS BIGINT)
    END AS Number_of_Discharges_numeric,

    CASE
        WHEN Predicted_Readmission_Rate = 'N/A' THEN NULL
        ELSE CAST(Predicted_Readmission_Rate AS DOUBLE)
    END AS Predicted_Readmission_Rate_numeric,
    
    CASE
        WHEN Expected_Readmission_Rate = 'N/A' THEN NULL
        ELSE CAST(Expected_Readmission_Rate AS DOUBLE)
    END AS Expected_Readmission_Rate_numeric


FROM cms_readmissions_clean2;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select Number_of_Readmissions, Number_of_Readmissions_numeric, readmissions_too_few_flag, Excess_Readmission_Ratio, excess_readmission_ratio_numeric
from workspace.default.cms_readmissions_clean3
limit 20

Number_of_Readmissions,Number_of_Readmissions_numeric,readmissions_too_few_flag,Excess_Readmission_Ratio,excess_readmission_ratio_numeric
Too Few to Report,null,1,0.9875,0.9875
13,13,0,0.9531,0.9531
33,33,0,0.9370,0.937
19,19,0,0.9823,0.9823
79,79,0,0.9871,0.9871
136,136,0,1.0233,1.0233
N/A,null,0,N/A,null
N/A,null,0,N/A,null
17,17,0,0.9308,0.9308
35,35,0,1.0087,1.0087


In [0]:
%sql

DESCRIBE workspace.default.cms_readmissions_clean3

col_name,data_type,comment
facility_name,string,null
facility_id,bigint,null
state,string,null
measure_name,string,null
Number_of_Discharges,string,null
Footnote,bigint,null
Excess_Readmission_Ratio,string,null
Predicted_Readmission_Rate,string,null
Expected_Readmission_Rate,string,null
Number_of_Readmissions,string,null


Records associated with CMS reporting footnotes were classified by reporting status so selected KPI calculations can later exclude or filter records with known reporting limitations.

In [0]:
%sql

SELECT
    Footnote, footnote_explanation,
    COUNT(*) AS row_count

FROM cms_readmissions_clean3

GROUP BY
    Footnote,
    footnote_explanation

ORDER BY row_count DESC;

Footnote,footnote_explanation,row_count
null,null,11343
5,Results are not available for this reporting period.,3255
1,The number of cases/patients is too few to report.,3150
29,This measure was calculated using partial performance period data due to a CMS-approved exception.,377
7,No cases met the criteria for this measure.,205


In [0]:
%sql

CREATE OR REPLACE TABLE cms_readmissions_clean4 AS

SELECT *,

-- Add a reporting status based on footnotes
    CASE
        WHEN Footnote IN (1,5,7)
            THEN 'Unavailable/Suppressed'

        WHEN Footnote = 29
            THEN 'Partial Reporting Period'

        ELSE 'Standard Reporting'
    END AS reporting_status

FROM cms_readmissions_clean3;




num_affected_rows,num_inserted_rows


## Data Availability and Reporting Status

In [0]:
%sql
select *
from workspace.default.cms_readmissions_clean4
limit 20

facility_name,facility_id,state,measure_name,Number_of_Discharges,Footnote,Excess_Readmission_Ratio,Predicted_Readmission_Rate,Expected_Readmission_Rate,Number_of_Readmissions,Start_Date,End_date,measure_display_name,footnote_number,footnote_Explanation,Number_of_Readmissions_numeric,readmissions_too_few_flag,excess_readmission_ratio_numeric,Number_of_Discharges_numeric,Predicted_Readmission_Rate_numeric,Expected_Readmission_Rate_numeric,reporting_status
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,N/A,null,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,2024-06-30,Hip/Knee Replacement,null,null,null,1,0.9875,null,4.5734,4.6311,Standard Reporting
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137,null,0.9531,10.3960,10.9078,13,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),null,null,13,0,0.9531,137,10.396,10.9078,Standard Reporting
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273,null,0.9370,13.2998,14.1948,33,07/01/2021,2024-06-30,Heart Attack,null,null,33,0,0.937,273,13.2998,14.1948,Standard Reporting
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122,null,0.9823,16.6384,16.9389,19,07/01/2021,2024-06-30,COPD,null,null,19,0,0.9823,122,16.6384,16.9389,Standard Reporting
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507,null,0.9871,15.7529,15.9591,79,07/01/2021,2024-06-30,Pneumonia,null,null,79,0,0.9871,507,15.7529,15.9591,Standard Reporting
SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HF-HRRP,653,null,1.0233,20.5695,20.1010,136,07/01/2021,2024-06-30,Heart Failure,null,null,136,0,1.0233,653,20.5695,20.101,Standard Reporting
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-CABG-HRRP,N/A,5,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30,Coronary Artery Bypass Graft Surgery (CABG),5,Results are not available for this reporting period.,null,0,null,null,null,null,Unavailable/Suppressed
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-AMI-HRRP,N/A,1,N/A,N/A,N/A,N/A,07/01/2021,2024-06-30,Heart Attack,1,The number of cases/patients is too few to report.,null,0,null,null,null,null,Unavailable/Suppressed
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-COPD-HRRP,132,null,0.9308,16.8541,18.1080,17,07/01/2021,2024-06-30,COPD,null,null,17,0,0.9308,132,16.8541,18.108,Standard Reporting
MARSHALL MEDICAL CENTERS,10005,AL,READM-30-HF-HRRP,164,null,1.0087,20.9512,20.7700,35,07/01/2021,2024-06-30,Heart Failure,null,null,35,0,1.0087,164,20.9512,20.77,Standard Reporting
